# [8.3] ACDC and Circuit Metrics - Exercises

**By the end of this notebook, you will have shown that iterative edge deletion with metric recomputation recovers a nonlinear ground-truth circuit which one-shot edge scores completely miss, then tested the same search discipline on named attention heads in a pinned TransformerLens model.**

```python
EXERCISE_ID = "8_3_acdc_and_circuit_metrics"
GT_TIER = "GT-0"
DIFFICULTY = 4
IMPORTANCE = 5
EXPECTED_RUNTIME = "about 90 minutes for the exercises; about 30 seconds for the cached CPU solution run"
REQUIRES_GPU = False
```

## Core Question

**Can an automated deletion search recover the edges that actually implement a behavior, including edges whose importance only appears when the rest of their path is present?**

Original ARENA's IOI sequence does not begin with a finished circuit. It moves from a crisp task metric, through exploratory evidence, to exact interventions, minimal circuits, and anomaly hunting. We will follow the same rhythm on a model organism where the answer is known before moving to a real model where it is not.

## Learning Objectives

1. Run a named directed acyclic graph and cache every node activation.
2. Implement edge interventions which replace removed messages with a corrupt-run reference.
3. Explain why isolated insertion scores fail on interacting paths.
4. Implement greedy ACDC deletion with metric recomputation after every decision.
5. Sweep the threshold and separate faithfulness, minimality, and completeness.
6. Compare against every same-size circuit, not one convenient random draw.
7. Patch named TransformerLens attention heads and interpret a component-level result honestly.

## Cold Open - the edge that looks useless

The primary path below contains a product gate. Both of its inputs are required. If you insert only one clean edge into an otherwise corrupt graph, the other input remains zero and the output does not move. Every single-edge insertion therefore scores **zero**, even though six primary edges are individually necessary in the complete graph.

That is the point of this lesson: **ACDC is a sequential intervention algorithm, not `scores > threshold`.**


In [ ]:
import json
import sys
from collections.abc import Callable, Mapping, Sequence
from dataclasses import asdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch as t

chapter = "chapter8_automated_circuits"
section = "part3_acdc_circuit_metrics"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
assets_dir = root_dir / chapter / "instructions" / "assets"
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_acdc_circuit_metrics.solutions as reference
import part3_acdc_circuit_metrics.tests as tests
from part3_acdc_circuit_metrics.solutions import (
    ACDCResult,
    ACDCStep,
    CircuitMetricsReport,
    Edge,
    GraphRun,
    OODCircuitReport,
    SameSizeCircuitReport,
    ThresholdSweepPoint,
    ToyCircuitGraph,
    TOY_CLEAN_INPUTS,
    TOY_CORRUPT_INPUTS,
    TOY_OOD_INPUTS,
    _apply_operation,
    _validate_edge_names,
    _validate_graph,
    _validate_inputs,
    build_toy_acdc_graph,
)

t.set_grad_enabled(False)
MetricEvaluator = Callable[[frozenset[str]], float]

graph = build_toy_acdc_graph()
edge_names = tuple(edge.name for edge in graph.edges)
print(f"{len(graph.node_order)} named nodes, {len(edge_names)} candidate edges")
print(f"ground truth: {len(graph.ground_truth_edges)} edges; controls: {len(graph.decoy_edges)} edges")


### Exercise - execute the named graph

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> Suggested time: 10 minutes
> ```

Implement `run_toy_graph`. Traverse `graph.node_order`, gather weighted messages from incoming edges, and apply each node's `sum` or `product` operation. Return the complete activation cache, not just the final scalar: the cache is what makes later interventions debuggable.

<details><summary>Expected output</summary>

```text
All tests in `test_toy_graph_forward_values` passed!
All tests in `test_toy_graph_rejects_bad_inputs` passed!
clean metric = 2.5; corrupt metric = 0.1; clean binding activation = 1.0
```

</details>

<details><summary>Help - Start from the input cache</summary>

Initialize the activation dictionary with the four input values. For each later node, select edges whose `receiver` is that node, multiply sender activation by edge weight, then apply `_apply_operation`.

</details>

<details><summary>Common bug</summary>

Computing only the output loses the intermediate node values needed for corrupt substitutions. Another common bug is evaluating nodes alphabetically rather than in topological order.

</details>

<details><summary>Solution</summary>

```python
def run_toy_graph(graph: ToyCircuitGraph, inputs: Mapping[str, float]) -> GraphRun:
    """Run every edge of the toy graph on one set of inputs."""

    _validate_graph(graph)
    _validate_inputs(graph, inputs)
    incoming = {
        node: tuple(edge for edge in graph.edges if edge.receiver == node)
        for node in graph.node_order
    }
    activations: dict[str, float] = {name: float(inputs[name]) for name in graph.input_nodes}
    for node in graph.node_order:
        if node in graph.input_nodes:
            continue
        messages = [activations[edge.sender] * edge.weight for edge in incoming[node]]
        activations[node] = _apply_operation(graph.operations[node], messages)
    return GraphRun(
        activations=activations,
        metric=activations[graph.output_node],
        active_edges=tuple(edge.name for edge in graph.edges),
    )
```

</details>


In [ ]:
def run_toy_graph(graph: ToyCircuitGraph, inputs: Mapping[str, float]) -> GraphRun:
    """Run every edge of the toy graph and return all node activations."""
    raise NotImplementedError()


tests.test_toy_graph_forward_values(run_toy_graph)
tests.test_toy_graph_rejects_bad_inputs(run_toy_graph)


### Exercise - intervene on edges, then recompute downstream nodes

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> Suggested time: 20 minutes
> ```

Implement `run_edge_intervention` and `normalized_recovery`. A kept edge transmits its sender's current hybrid activation. A removed edge transmits the sender activation cached in the fully corrupt run. Recompute every downstream node in topological order.

This is closer to path patching than zero ablation: an absent circuit edge gets a task-matched corrupt value, which keeps the intervention on-distribution in the toy model.

<details><summary>Expected output</summary>

```text
All tests in `test_edge_intervention_endpoints` passed!
All tests in `test_edge_intervention_recomputes_downstream` passed!
All tests in `test_normalized_recovery` passed!
Removing one product-gate input leaves metric 0.5 and recovery 1/6.
```

</details>

<details><summary>Help - Cache corrupt first</summary>

Run the complete corrupt graph once. At each hybrid edge choose `activations[sender]` when kept and `corrupt.activations[sender]` when removed. Normalize with `(metric - corrupt) / (clean - corrupt)`.

</details>

<details><summary>Common bug</summary>

Replacing a missing edge with zero changes the reference distribution. Replacing a kept edge with the clean cache also freezes its upstream computation; kept edges should transmit the current hybrid sender value.

</details>

<details><summary>Solution</summary>

```python
def run_edge_intervention(
    graph: ToyCircuitGraph,
    clean_inputs: Mapping[str, float],
    corrupt_inputs: Mapping[str, float],
    active_edges: Sequence[str] | set[str] | frozenset[str],
) -> GraphRun:
    """Run clean inputs while replacing missing edge messages with corrupt ones."""

    _validate_graph(graph)
    _validate_inputs(graph, clean_inputs)
    _validate_inputs(graph, corrupt_inputs)
    known_edges = {edge.name for edge in graph.edges}
    active = frozenset(active_edges)
    unknown = active - known_edges
    if unknown:
        raise ValueError(f"unknown active edges: {sorted(unknown)}")
    corrupt = run_toy_graph(graph, corrupt_inputs)
    incoming = {
        node: tuple(edge for edge in graph.edges if edge.receiver == node)
        for node in graph.node_order
    }
    activations: dict[str, float] = {
        name: float(clean_inputs[name]) for name in graph.input_nodes
    }
    for node in graph.node_order:
        if node in graph.input_nodes:
            continue
        messages = []
        for edge in incoming[node]:
            sender_value = (
                activations[edge.sender]
                if edge.name in active
                else corrupt.activations[edge.sender]
            )
            messages.append(sender_value * edge.weight)
        activations[node] = _apply_operation(graph.operations[node], messages)
    ordered_active = tuple(edge.name for edge in graph.edges if edge.name in active)
    return GraphRun(activations, activations[graph.output_node], ordered_active)

def normalized_recovery(*, clean_metric: float, corrupt_metric: float, metric: float) -> float:
    """Normalize a circuit metric so corrupt is zero and clean is one."""

    values = t.tensor([clean_metric, corrupt_metric, metric], dtype=t.float64)
    if not t.isfinite(values).all():
        raise ValueError("clean, corrupt, and circuit metrics must be finite.")
    denominator = clean_metric - corrupt_metric
    if abs(denominator) < 1e-12:
        raise ValueError("clean_metric and corrupt_metric must differ.")
    return float((metric - corrupt_metric) / denominator)

def make_toy_evaluator(
    graph: ToyCircuitGraph,
    clean_inputs: Mapping[str, float] = TOY_CLEAN_INPUTS,
    corrupt_inputs: Mapping[str, float] = TOY_CORRUPT_INPUTS,
) -> tuple[MetricEvaluator, float, float]:
    clean_metric = run_toy_graph(graph, clean_inputs).metric
    corrupt_metric = run_toy_graph(graph, corrupt_inputs).metric

    def evaluate(active_edges: frozenset[str]) -> float:
        return run_edge_intervention(
            graph,
            clean_inputs,
            corrupt_inputs,
            active_edges,
        ).metric

    return evaluate, clean_metric, corrupt_metric
```

</details>


In [ ]:
def run_edge_intervention(
    graph: ToyCircuitGraph,
    clean_inputs: Mapping[str, float],
    corrupt_inputs: Mapping[str, float],
    active_edges: Sequence[str] | set[str] | frozenset[str],
) -> GraphRun:
    """Run clean inputs while removed edges transmit corrupt sender values."""
    raise NotImplementedError()


def normalized_recovery(*, clean_metric: float, corrupt_metric: float, metric: float) -> float:
    """Normalize corrupt to zero and clean to one."""
    raise NotImplementedError()


def make_toy_evaluator(graph, clean_inputs=TOY_CLEAN_INPUTS, corrupt_inputs=TOY_CORRUPT_INPUTS):
    clean_metric = run_toy_graph(graph, clean_inputs).metric
    corrupt_metric = run_toy_graph(graph, corrupt_inputs).metric
    def evaluate(active_edges):
        return run_edge_intervention(graph, clean_inputs, corrupt_inputs, active_edges).metric
    return evaluate, clean_metric, corrupt_metric


tests.test_edge_intervention_endpoints(run_edge_intervention)
tests.test_edge_intervention_recomputes_downstream(run_edge_intervention)
tests.test_normalized_recovery(normalized_recovery)


### Exercise - falsify one-shot edge scoring

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> Suggested time: 10 minutes
> ```

Implement `one_shot_insertion_scores`: start from the corrupt graph and activate one edge at a time. This is intentionally a method we expect to fail. The failure is the signature anomaly, not an inconvenience to hide.

<details><summary>Expected output</summary>

```text
All tests in `test_one_shot_scores_expose_interaction_failure` passed!
All ten isolated insertion scores are exactly 0.0.
A thresholded one-shot circuit therefore contains zero edges and has zero recovery.
```

</details>

<details><summary>Help - One singleton per edge</summary>

For each edge call `evaluate(frozenset({edge}))`, then normalize against the clean and corrupt endpoints.

</details>

<details><summary>Common bug</summary>

Scoring `all_edges - {edge}` computes deletion damage, not one-shot insertion. Both are useful, but they answer different questions.

</details>

<details><summary>Solution</summary>

```python
def one_shot_insertion_scores(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
) -> Mapping[str, float]:
    """Score each edge alone in the otherwise corrupt graph."""

    names = _validate_edge_names(edge_names)
    return {
        edge: normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=evaluate(frozenset({edge})),
        )
        for edge in names
    }
```

</details>


In [ ]:
def one_shot_insertion_scores(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
) -> Mapping[str, float]:
    raise NotImplementedError()


tests.test_one_shot_scores_expose_interaction_failure(one_shot_insertion_scores)

evaluator, clean_metric, corrupt_metric = make_toy_evaluator(graph)
one_shot = one_shot_insertion_scores(
    edge_names, evaluator, clean_metric=clean_metric, corrupt_metric=corrupt_metric
)
print({name: round(score, 3) for name, score in one_shot.items()})


### Exercise - implement greedy ACDC with recomputation

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
> Suggested time: 30 minutes
> ```

Implement `initial_deletion_order` and `greedy_acdc`.

Start with every edge. For each candidate, evaluate the **current circuit minus that edge**. Remove it only when the normalized damage is at most the threshold. If you remove it, the next candidate must be tested in the already-pruned graph. Record every decision in `ACDCStep` so a learner can inspect the search rather than receiving a final edge list from a black box.

<details><summary>Expected output</summary>

```text
All tests in `test_initial_order_puts_decoys_first` passed!
All tests in `test_greedy_acdc_recovers_ground_truth` passed!
All tests in `test_greedy_acdc_recomputes_after_each_removal` passed!
Kept: 8/10 exact ground-truth edges; recovery: 1.000.
```

</details>

<details><summary>Help - Cache the accepted state, not every trial</summary>

Maintain `active`, `current_metric`, and `current_recovery`. A failed deletion leaves them unchanged; an accepted deletion replaces all three with the trial values.

</details>

<details><summary>Common bug</summary>

Re-evaluating every candidate against the original full graph silently turns the algorithm back into static score thresholding. The recording test checks that later calls see the cumulatively pruned edge set.

</details>

<details><summary>Solution</summary>

```python
def initial_deletion_order(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
) -> tuple[str, ...]:
    """Rank least damaging full-circuit deletions first."""

    names = _validate_edge_names(edge_names)
    full = frozenset(names)
    full_recovery = normalized_recovery(
        clean_metric=clean_metric,
        corrupt_metric=corrupt_metric,
        metric=evaluate(full),
    )
    damages = {}
    for edge in names:
        trial_recovery = normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=evaluate(full - {edge}),
        )
        damages[edge] = full_recovery - trial_recovery
    return tuple(sorted(names, key=lambda edge: (damages[edge], names.index(edge))))

def greedy_acdc(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
    threshold: float,
    order: Sequence[str] | None = None,
) -> ACDCResult:
    """Delete edges greedily, recomputing the metric after every decision."""

    names = _validate_edge_names(edge_names)
    if not t.isfinite(t.tensor(threshold)).item() or threshold < 0:
        raise ValueError("threshold must be finite and nonnegative.")
    deletion_order = tuple(order) if order is not None else names
    if len(deletion_order) != len(names) or set(deletion_order) != set(names):
        raise ValueError("order must be a permutation of edge_names.")
    active = frozenset(names)
    current_metric = evaluate(active)
    current_recovery = normalized_recovery(
        clean_metric=clean_metric,
        corrupt_metric=corrupt_metric,
        metric=current_metric,
    )
    steps: list[ACDCStep] = []
    removed: list[str] = []
    for edge in deletion_order:
        before_recovery = current_recovery
        trial_active = active - {edge}
        trial_metric = evaluate(trial_active)
        trial_recovery = normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=trial_metric,
        )
        damage = before_recovery - trial_recovery
        if damage <= threshold:
            decision = "remove"
            active = trial_active
            current_metric = trial_metric
            current_recovery = trial_recovery
            removed.append(edge)
        else:
            decision = "keep"
        steps.append(ACDCStep(edge, before_recovery, trial_recovery, damage, decision))
    kept = tuple(edge for edge in names if edge in active)
    return ACDCResult(kept, tuple(removed), threshold, current_recovery, tuple(steps))
```

</details>


In [ ]:
def initial_deletion_order(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *, clean_metric: float, corrupt_metric: float,
) -> tuple[str, ...]:
    raise NotImplementedError()


def greedy_acdc(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *, clean_metric: float, corrupt_metric: float,
    threshold: float,
    order: Sequence[str] | None = None,
) -> ACDCResult:
    raise NotImplementedError()


tests.test_initial_order_puts_decoys_first(initial_deletion_order)
tests.test_greedy_acdc_recovers_ground_truth(greedy_acdc)
tests.test_greedy_acdc_recomputes_after_each_removal(greedy_acdc)
tests.test_greedy_acdc_rejects_invalid_search(greedy_acdc)

deletion_order = initial_deletion_order(
    edge_names, evaluator, clean_metric=clean_metric, corrupt_metric=corrupt_metric
)
acdc = greedy_acdc(
    edge_names, evaluator,
    clean_metric=clean_metric, corrupt_metric=corrupt_metric,
    threshold=0.1, order=deletion_order,
)
for step in acdc.steps:
    print(f"{step.decision:>6}  damage={step.normalized_damage:6.3f}  {step.edge}")
print(f"kept={len(acdc.kept_edges)}/{len(edge_names)}, recovery={acdc.recovery:.3f}")


### Exercise - sweep thresholds and audit three circuit metrics

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> Suggested time: 25 minutes
> ```

Implement a full threshold sweep, rerunning the search at every threshold. Then implement:

- **faithfulness:** circuit recovery relative to clean and corrupt;
- **minimality:** damage from deleting each retained edge;
- **completeness:** gain from adding omitted subsets of up to two edges.

Why pairs? A missing two-edge path is invisible if completeness adds only one omitted edge at a time. The toy backup path makes this failure exact.

<details><summary>Expected output</summary>

```text
All tests in `test_threshold_sweep_shows_two_failure_cliffs` passed!
All tests in `test_circuit_metrics_are_distinct` passed!
All tests in `test_circuit_metrics_reject_incomplete_circuit` passed!
threshold 0.10 -> 8 edges / 1.000 recovery
threshold 0.17 -> 6 edges / 0.833 recovery
threshold 0.84 -> 0 edges / 0.000 recovery
```

</details>

<details><summary>Help - Rerun; do not filter one trace</summary>

Each threshold changes earlier decisions, so call `greedy_acdc` from a fresh full edge set. For completeness, enumerate omitted subsets of sizes one and two and keep the largest recovery gain.

</details>

<details><summary>Common bug</summary>

Adding only one omitted edge declares the primary-only circuit complete, because neither half of the two-edge backup path works alone. This is the same interaction trap as the cold open.

</details>

<details><summary>Solution</summary>

```python
def threshold_sweep(
    edge_names: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
    thresholds: Sequence[float],
    ground_truth_edges: Sequence[str],
    order: Sequence[str] | None = None,
) -> tuple[ThresholdSweepPoint, ...]:
    """Run the full deletion search independently at every threshold."""

    if not thresholds:
        raise ValueError("thresholds must be nonempty.")
    truth = set(ground_truth_edges)
    points = []
    for threshold in thresholds:
        result = greedy_acdc(
            edge_names,
            evaluate,
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            threshold=float(threshold),
            order=order,
        )
        points.append(
            ThresholdSweepPoint(
                float(threshold),
                len(result.kept_edges),
                result.recovery,
                set(result.kept_edges) == truth,
            )
        )
    return tuple(points)

def evaluate_circuit_metrics(
    edge_names: Sequence[str],
    circuit_edges: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
    min_faithfulness: float = 0.95,
    min_edge_damage: float = 0.05,
    max_omitted_gain: float = 0.05,
    max_completeness_subset_size: int = 2,
) -> CircuitMetricsReport:
    """Measure faithfulness, edgewise minimality, and subset completeness."""

    names = _validate_edge_names(edge_names)
    circuit = frozenset(circuit_edges)
    if not circuit or not circuit.issubset(names):
        raise ValueError("circuit_edges must be a nonempty subset of edge_names.")
    for label, value in {
        "min_faithfulness": min_faithfulness,
        "min_edge_damage": min_edge_damage,
        "max_omitted_gain": max_omitted_gain,
    }.items():
        if not t.isfinite(t.tensor(value)).item() or value < 0:
            raise ValueError(f"{label} must be finite and nonnegative.")
    if max_completeness_subset_size <= 0:
        raise ValueError("max_completeness_subset_size must be positive.")

    def recovery(active: frozenset[str]) -> float:
        return normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=evaluate(active),
        )

    circuit_recovery = recovery(circuit)
    minimality = {
        edge: circuit_recovery - recovery(circuit - {edge})
        for edge in names
        if edge in circuit
    }
    omitted = tuple(edge for edge in names if edge not in circuit)
    completeness = {edge: recovery(circuit | {edge}) - circuit_recovery for edge in omitted}
    subset_rows: dict[tuple[str, ...], float] = {}
    for subset_size in range(1, min(max_completeness_subset_size, len(omitted)) + 1):
        for subset in combinations(omitted, subset_size):
            subset_rows[subset] = recovery(circuit | set(subset)) - circuit_recovery
    min_damage = min(minimality.values())
    strongest_subset = max(subset_rows, key=subset_rows.get) if subset_rows else ()
    max_gain = subset_rows.get(strongest_subset, 0.0)
    return CircuitMetricsReport(
        circuit_recovery,
        minimality,
        min_damage,
        completeness,
        {" + ".join(subset): gain for subset, gain in subset_rows.items()},
        strongest_subset,
        max_gain,
        circuit_recovery >= min_faithfulness,
        min_damage >= min_edge_damage,
        max_gain <= max_omitted_gain,
    )
```

</details>


In [ ]:
def threshold_sweep(
    edge_names, evaluate, *, clean_metric, corrupt_metric,
    thresholds, ground_truth_edges, order=None,
) -> tuple[ThresholdSweepPoint, ...]:
    raise NotImplementedError()


def evaluate_circuit_metrics(
    edge_names, circuit_edges, evaluate, *, clean_metric, corrupt_metric,
    min_faithfulness=0.95, min_edge_damage=0.05,
    max_omitted_gain=0.05, max_completeness_subset_size=2,
) -> CircuitMetricsReport:
    raise NotImplementedError()


tests.test_threshold_sweep_shows_two_failure_cliffs(threshold_sweep)
tests.test_circuit_metrics_are_distinct(evaluate_circuit_metrics)
tests.test_circuit_metrics_reject_incomplete_circuit(evaluate_circuit_metrics)

thresholds = (0.0, 0.05, 0.10, 0.16, 0.17, 0.40, 0.83, 0.84)
sweep = threshold_sweep(
    edge_names, evaluator, clean_metric=clean_metric, corrupt_metric=corrupt_metric,
    thresholds=thresholds, ground_truth_edges=graph.ground_truth_edges,
    order=deletion_order,
)
metrics = evaluate_circuit_metrics(
    edge_names, acdc.kept_edges, evaluator,
    clean_metric=clean_metric, corrupt_metric=corrupt_metric,
)
print([(p.threshold, p.circuit_size, round(p.recovery, 3)) for p in sweep])
print(f"faithfulness={metrics.circuit_recovery:.3f}, min damage={metrics.min_edge_damage:.3f}, max omitted gain={metrics.max_omitted_gain:.3f}")


### Exercise - beat every same-size circuit and held-out toy regime

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> Suggested time: 20 minutes
> ```

Implement `same_size_circuit_report` by enumerating all `10 choose 8 = 45` circuits, and implement `evaluate_toy_ood` on three unseen signal scales. Enumeration turns a noisy random baseline into an exact null distribution.

Rank is more informative than a cherry-picked comparison: the discovered circuit should be the unique best same-size graph.

<details><summary>Expected output</summary>

```text
All tests in `test_same_size_controls_are_exact` passed!
All tests in `test_toy_ood_preserves_exact_circuit` passed!
discovered rank = 1 / 45; exact p = 0.0222
best wrong recovery = 0.8333; mean wrong recovery = 0.1970
worst held-out recovery = 1.000
```

</details>

<details><summary>Help - Enumerate combinations of names</summary>

Use `itertools.combinations(edge_names, len(circuit_edges))`. Normalize every candidate's metric, keep the discovered candidate in the rank calculation, and exclude it from control mean and best values.

</details>

<details><summary>Common bug</summary>

Sampling one random circuit can make the result depend on a lucky seed. Comparing circuits of different sizes also confounds discovery quality with intervention strength.

</details>

<details><summary>Solution</summary>

```python
def same_size_circuit_report(
    edge_names: Sequence[str],
    circuit_edges: Sequence[str],
    evaluate: MetricEvaluator,
    *,
    clean_metric: float,
    corrupt_metric: float,
) -> SameSizeCircuitReport:
    """Enumerate every same-size circuit, an exact random-circuit null."""

    names = _validate_edge_names(edge_names)
    circuit = frozenset(circuit_edges)
    if not circuit or not circuit.issubset(names):
        raise ValueError("circuit_edges must be a nonempty subset of edge_names.")
    all_rows: list[tuple[frozenset[str], float]] = []
    for candidate in combinations(names, len(circuit)):
        candidate_set = frozenset(candidate)
        recovery = normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=evaluate(candidate_set),
        )
        all_rows.append((candidate_set, recovery))
    discovered = next(value for candidate, value in all_rows if candidate == circuit)
    controls = [value for candidate, value in all_rows if candidate != circuit]
    rank = 1 + sum(value > discovered + 1e-12 for _, value in all_rows)
    pvalue = sum(value >= discovered - 1e-12 for _, value in all_rows) / len(all_rows)
    return SameSizeCircuitReport(
        len(circuit),
        len(all_rows),
        discovered,
        rank,
        pvalue,
        float(sum(controls) / len(controls)) if controls else 0.0,
        float(max(controls)) if controls else discovered,
        tuple(float(value) for value in controls),
    )

def evaluate_toy_ood(
    graph: ToyCircuitGraph,
    circuit_edges: Sequence[str],
    templates: Sequence[tuple[str, Mapping[str, float], Mapping[str, float]]] = TOY_OOD_INPUTS,
    *,
    min_recovery: float = 0.95,
) -> OODCircuitReport:
    """Evaluate the exact circuit on held-out signal scales and nuisance values."""

    if not templates:
        raise ValueError("templates must be nonempty.")
    recoveries = {}
    for name, clean_inputs, corrupt_inputs in templates:
        evaluator, clean_metric, corrupt_metric = make_toy_evaluator(
            graph, clean_inputs, corrupt_inputs
        )
        recoveries[name] = normalized_recovery(
            clean_metric=clean_metric,
            corrupt_metric=corrupt_metric,
            metric=evaluator(frozenset(circuit_edges)),
        )
    worst = min(recoveries.values())
    return OODCircuitReport(recoveries, worst, worst >= min_recovery)
```

</details>


In [ ]:
def same_size_circuit_report(
    edge_names, circuit_edges, evaluate, *, clean_metric, corrupt_metric,
) -> SameSizeCircuitReport:
    raise NotImplementedError()


def evaluate_toy_ood(
    graph, circuit_edges, templates=TOY_OOD_INPUTS, *, min_recovery=0.95,
) -> OODCircuitReport:
    raise NotImplementedError()


tests.test_same_size_controls_are_exact(same_size_circuit_report)
tests.test_toy_ood_preserves_exact_circuit(evaluate_toy_ood)

random_report = same_size_circuit_report(
    edge_names, acdc.kept_edges, evaluator,
    clean_metric=clean_metric, corrupt_metric=corrupt_metric,
)
ood_report = evaluate_toy_ood(graph, acdc.kept_edges)
print(f"rank={random_report.discovered_rank}/{random_report.num_circuits}, p={random_report.exact_empirical_pvalue:.4f}")
print(f"wrong mean={random_report.control_mean_recovery:.3f}, wrong best={random_report.control_best_recovery:.3f}")
print({name: round(value, 3) for name, value in ood_report.recoveries.items()})


## Signature Result - a circuit, a threshold curve, and an exact null

The figure must answer three questions at a glance:

1. **What did ACDC keep?** Eight named causal edges, with the two matched background edges removed.
2. **How brittle is the threshold?** The weak backup disappears near `1/6`; the primary nonlinear path disappears near `5/6`.
3. **Could a same-size circuit do this by accident?** Only one of all 45 candidates reaches full recovery.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
fig.patch.set_facecolor("#f7f7f4")

G = nx.DiGraph()
for edge in graph.edges:
    G.add_edge(edge.sender, edge.receiver, name=edge.name)
pos = {
    "tokens.io_name": (0, 3.0), "tokens.position": (0, 2.0),
    "tokens.backup": (0, 1.0), "tokens.background": (0, 0.0),
    "L0H0.name_copy": (1, 3.0), "L0H1.position": (1, 2.0),
    "L0H2.background": (1, 0.0), "L0M0.binding": (2, 2.5),
    "L1M0.backup": (2, 1.0), "L1H0.answer": (3, 2.5),
    "logits.io": (4, 1.8),
}
labels = {
    "tokens.io_name": "IO name", "tokens.position": "position",
    "tokens.backup": "backup cue", "tokens.background": "background",
    "L0H0.name_copy": "L0H0\nname copy", "L0H1.position": "L0H1\nposition",
    "L0H2.background": "L0H2\nbackground", "L0M0.binding": "L0M0\nproduct gate",
    "L1M0.backup": "L1M0\nbackup", "L1H0.answer": "L1H0\nanswer",
    "logits.io": "IO logit",
}
node_colors = ["#d9edf2" if node.startswith("tokens") else "#f4d7a1" if "M0" in node else "#d8e7c5" for node in G.nodes]
edge_colors = ["#187a79" if data["name"] in acdc.kept_edges else "#b4b4b4" for _, _, data in G.edges(data=True)]
edge_styles = ["solid" if data["name"] in acdc.kept_edges else "dashed" for _, _, data in G.edges(data=True)]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, edgecolors="#333333", linewidths=0.8, ax=axes[0])
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=axes[0])
for style in ("solid", "dashed"):
    chosen = [(u, v) for (u, v, data), s in zip(G.edges(data=True), edge_styles, strict=True) if s == style]
    colors = [edge_colors[list(G.edges).index((u, v))] for u, v in chosen]
    nx.draw_networkx_edges(G, pos, edgelist=chosen, edge_color=colors, style=style, width=2.2, arrowsize=15, ax=axes[0])
axes[0].set_title("Exact circuit recovered", fontweight="bold")
axes[0].set_xlim(-0.35, 4.45)
axes[0].set_ylim(-0.75, 3.45)
axes[0].text(0.0, -0.55, "teal = kept  |  dashed = corrupt-matched decoy", fontsize=8)
axes[0].axis("off")

x = [point.threshold for point in sweep]
recovery_y = [point.recovery for point in sweep]
size_y = [point.circuit_size for point in sweep]
axes[1].plot(x, recovery_y, marker="o", color="#187a79", linewidth=2.5, label="faithfulness")
axes[1].axhline(0.95, color="#b23a48", linestyle="--", linewidth=1.2, label="95% target")
axes[1].set_xlabel("deletion threshold")
axes[1].set_ylabel("normalized recovery")
axes[1].set_ylim(-0.04, 1.08)
ax_size = axes[1].twinx()
ax_size.step(x, size_y, where="mid", color="#d28e2c", linewidth=2, label="circuit size")
ax_size.set_ylabel("kept edges")
ax_size.set_ylim(-0.4, 10.4)
axes[1].set_title("Threshold reveals two mechanisms", fontweight="bold")
axes[1].legend(loc="lower left", frameon=False)
ax_size.legend(loc="center left", frameon=False)

axes[2].hist(random_report.control_recoveries, bins=np.linspace(-0.02, 1.02, 13), color="#9ea7ad", edgecolor="white")
axes[2].axvline(random_report.discovered_recovery, color="#187a79", linewidth=3, label="ACDC circuit")
axes[2].axvline(random_report.control_best_recovery, color="#d28e2c", linestyle="--", linewidth=2, label="best wrong")
axes[2].set_xlabel("normalized recovery")
axes[2].set_ylabel("same-size circuits")
axes[2].set_title("Exact same-size null", fontweight="bold")
axes[2].legend(frameon=False)
axes[2].text(0.03, 0.92, f"rank 1/{random_report.num_circuits}\nexact p={random_report.exact_empirical_pvalue:.3f}", transform=axes[2].transAxes, va="top", fontweight="bold")

toy_asset = assets_dir / "acdc_toy_ground_truth_signature.png"
fig.savefig(toy_asset, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


<details><summary>Interpretation - why the result is convincing</summary>

The exact oracle says which edges are causal before search begins. ACDC returns exactly those eight edges, preserves all of the clean-corrupt gap, finds every kept edge necessary, finds no gain from the omitted decoy pair, and ranks first among all same-size graphs. The one-shot baseline returns an empty circuit. This is not a pretty graph with a story attached; it is a falsifiable algorithmic result.

</details>

<details><summary>Interpretation - why completeness tests pairs</summary>

If we deliberately omit the two-edge backup path, adding either edge alone does nothing. A single-edge completeness check therefore says "complete" when the circuit is missing one sixth of the behavior. Testing omitted pairs catches the missing path. In real models the combinatorics become expensive, which is exactly why completeness claims need an explicit search scope.

</details>


## Real-Model Path - named heads feeding a real MLP

Our clean prompt is `"The cat sat on the"`; the model prefers `" floor"`. The corrupt prompt is `"The bird flew over the"`; it prefers `" top"`. The signed metric is `logit(" floor") - logit(" top")`.

We run the same greedy deletion routine over `L0H0` through `L0H7`. Kept heads are patched clean into the corrupt run; all others stay corrupt; the real `L0MLP` recomputes on every trial. We also enumerate all 56 three-head sets and test the discovered set on three held-out prompt pairs.

This is an **honest component-level result**, not full edge-level ACDC. Head outputs are candidate components, and the MLP is measured as a downstream bottleneck. We do not claim an IOI or greater-than circuit.


### Exercise - patch named attention-head results

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> Suggested time: 15 minutes
> ```

Now move to `NeelNanda/GELU_1L512W_C4_Code`, a pinned one-layer TransformerLens model with eight heads and one MLP. Implement the hook helper which replaces selected slices of `blocks.0.attn.hook_result` with their clean cached values.

We patch all positions for selected heads. The unpatched heads remain corrupt, and the real MLP recomputes from the hybrid residual stream.

<details><summary>Expected output</summary>

```text
All tests in `test_patch_selected_head_results` passed!
All tests in `test_patch_selected_head_results_rejects_bad_heads` passed!
Only the selected head axis slices change; the input tensor is not mutated.
```

</details>

<details><summary>Help - The head axis is dimension two</summary>

`hook_result` has shape `[batch, position, head, d_model]`. Clone the corrupt activation, then assign clean values at `[:, :, selected_heads, :]`.

</details>

<details><summary>Common bug</summary>

Patching the last axis treats residual dimensions as heads. Mutating the hook input in place can also leak state into later interventions.

</details>

<details><summary>Solution</summary>

```python
def patch_selected_head_results(
    activation: t.Tensor,
    clean_head_results: t.Tensor,
    selected_heads: Sequence[int],
) -> t.Tensor:
    """Patch selected clean head results into a corrupt hook activation."""

    if activation.ndim != 4:
        raise ValueError("head result must have shape [batch, position, head, d_model].")
    if activation.shape != clean_head_results.shape:
        raise ValueError("clean and corrupt head-result tensors must have matching shapes.")
    if not t.isfinite(activation).all() or not t.isfinite(clean_head_results).all():
        raise ValueError("head-result tensors must be finite.")
    heads = tuple(int(head) for head in selected_heads)
    if len(set(heads)) != len(heads):
        raise ValueError("selected_heads must not contain duplicates.")
    if any(head < 0 or head >= activation.shape[2] for head in heads):
        raise ValueError("selected head index is out of range.")
    patched = activation.clone()
    if heads:
        patched[:, :, list(heads), :] = clean_head_results[:, :, list(heads), :]
    return patched
```

</details>


In [ ]:
def patch_selected_head_results(
    activation: t.Tensor,
    clean_head_results: t.Tensor,
    selected_heads: Sequence[int],
) -> t.Tensor:
    raise NotImplementedError()


tests.test_patch_selected_head_results(patch_selected_head_results)
tests.test_patch_selected_head_results_rejects_bad_heads(patch_selected_head_results)


In [ ]:
model = reference.load_pinned_gelu1l(device="cpu")
task = reference._prepare_transformerlens_task(model, *reference.TL_PRIMARY_PAIR)
head_names = tuple(f"L0H{head}" for head in range(model.cfg.n_heads))

def make_head_evaluator(model, task):
    def evaluate(active_heads: frozenset[str]) -> float:
        selected = tuple(int(name.removeprefix("L0H")) for name in active_heads)
        def patch_hook(activation, hook):
            del hook
            return patch_selected_head_results(activation, task["clean_head_results"], selected)
        logits = model.run_with_hooks(
            task["corrupt_tokens"],
            fwd_hooks=[(reference.TL_HEAD_HOOK, patch_hook)],
        )
        return reference.answer_logit_diff(
            logits,
            positive_token_id=task["positive_token_id"],
            negative_token_id=task["negative_token_id"],
        )
    return evaluate

head_evaluator = make_head_evaluator(model, task)
head_order = initial_deletion_order(
    head_names, head_evaluator,
    clean_metric=task["clean_metric"], corrupt_metric=task["corrupt_metric"],
)
head_acdc = greedy_acdc(
    head_names, head_evaluator,
    clean_metric=task["clean_metric"], corrupt_metric=task["corrupt_metric"],
    threshold=0.05, order=head_order,
)
head_random = same_size_circuit_report(
    head_names, head_acdc.kept_edges, head_evaluator,
    clean_metric=task["clean_metric"], corrupt_metric=task["corrupt_metric"],
)
all_heads = frozenset(head_names)
all_head_recovery = normalized_recovery(
    clean_metric=task["clean_metric"], corrupt_metric=task["corrupt_metric"],
    metric=head_evaluator(all_heads),
)
head_deletion_damage = {
    name: all_head_recovery - normalized_recovery(
        clean_metric=task["clean_metric"], corrupt_metric=task["corrupt_metric"],
        metric=head_evaluator(all_heads - {name}),
    )
    for name in head_names
}

heldout_recoveries = {}
for label, clean_prompt, corrupt_prompt in reference.TL_HELDOUT_PAIRS:
    heldout_task = reference._prepare_transformerlens_task(model, clean_prompt, corrupt_prompt)
    heldout_eval = make_head_evaluator(model, heldout_task)
    heldout_recoveries[label] = normalized_recovery(
        clean_metric=heldout_task["clean_metric"],
        corrupt_metric=heldout_task["corrupt_metric"],
        metric=heldout_eval(frozenset(head_acdc.kept_edges)),
    )

mlp_bottleneck_recovery = reference._mlp_bottleneck_recovery(model, task)
real_result = {
    "positive_token": task["positive_token"],
    "negative_token": task["negative_token"],
    "clean_corrupt_gap": task["clean_metric"] - task["corrupt_metric"],
    "kept_heads": head_acdc.kept_edges,
    "primary_recovery": head_acdc.recovery,
    "same_size_rank": head_random.discovered_rank,
    "same_size_total": head_random.num_circuits,
    "random_mean": head_random.control_mean_recovery,
    "random_best": head_random.control_best_recovery,
    "heldout_recoveries": heldout_recoveries,
    "mlp_bottleneck_recovery": mlp_bottleneck_recovery,
}
print(json.dumps(real_result, indent=2))


<details><summary>Expected output - pinned CPU result</summary>

```text
target tokens: " floor" vs " top"
clean-corrupt gap: 6.2771
kept heads: [L0H0, L0H4, L0H6]
primary recovery: 0.9455
same-size rank: 1 / 56
wrong-set mean: 0.4095
best wrong set: 0.8855
held-out recovery range: 0.7855 to 0.9449
MLP clean-output recovery: 0.9912
```

</details>


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4), constrained_layout=True)
fig.patch.set_facecolor("#f7f7f4")

colors = ["#187a79" if name in head_acdc.kept_edges else "#a9afb3" for name in head_names]
axes[0].bar(head_names, [head_deletion_damage[name] for name in head_names], color=colors)
axes[0].axhline(0, color="#333333", linewidth=0.8)
axes[0].axhline(0.05, color="#b23a48", linestyle="--", linewidth=1.2, label="threshold")
axes[0].set_ylabel("full-set deletion damage")
axes[0].set_title("Named head candidates", fontweight="bold")
axes[0].legend(frameon=False)

axes[1].hist(head_random.control_recoveries, bins=np.linspace(0, 1, 13), color="#9ea7ad", edgecolor="white")
axes[1].axvline(head_acdc.recovery, color="#187a79", linewidth=3, label="L0H0 + L0H4 + L0H6")
axes[1].axvline(head_random.control_best_recovery, color="#d28e2c", linestyle="--", linewidth=2, label="best other set")
axes[1].set_xlabel("normalized recovery")
axes[1].set_ylabel("three-head sets")
axes[1].set_title("All 56 same-size sets", fontweight="bold")
axes[1].legend(frameon=False, fontsize=8)

labels = ["primary", *heldout_recoveries.keys(), "MLP only"]
values = [head_acdc.recovery, *heldout_recoveries.values(), mlp_bottleneck_recovery]
bar_colors = ["#187a79", "#5d95a3", "#5d95a3", "#5d95a3", "#d28e2c"]
axes[2].barh(labels, values, color=bar_colors)
axes[2].axvline(0.75, color="#b23a48", linestyle="--", linewidth=1.2)
axes[2].set_xlim(0, 1.08)
axes[2].set_xlabel("normalized recovery")
axes[2].set_title("Transfer and bottleneck control", fontweight="bold")
axes[2].text(0.755, 0.96, "held-out floor", transform=axes[2].get_xaxis_transform(),
             rotation=90, va="top", ha="left", color="#8d2936", fontsize=8)
for i, value in enumerate(values):
    axes[2].text(value + 0.015, i, f"{value:.3f}", va="center", fontsize=8)

real_asset = assets_dir / "acdc_transformerlens_component_signature.png"
fig.savefig(real_asset, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


<details><summary>Interpretation - what the real result supports</summary>

At threshold `0.05`, sequential deletion retains `L0H0`, `L0H4`, and `L0H6`. Together they recover `94.6%` of the primary clean-corrupt gap, rank first among all 56 three-head sets, and recover `78.5%-94.5%` on three held-out prompt pairs. This is evidence that these named head outputs carry reusable context information for this small continuation contrast.

</details>

<details><summary>Interpretation - the MLP anomaly</summary>

Patching the clean `L0MLP` output alone recovers `99.1%`, more than the selected heads. This does **not** make the heads irrelevant. A downstream patch can overwrite the consequence of upstream computation. The head search recomputes the MLP from hybrid head outputs; the MLP-only patch replaces that computation wholesale. The mismatch is a useful warning that component patching and edge-level causal stories are not interchangeable.

</details>

<details><summary>Help - why negative deletion damage is allowed</summary>

Removing `L0H2` slightly improves recovery. Real models contain inhibitory, compensatory, and task-irrelevant components, so an intervention score need not be positive. Greedy ACDC removes negative-damage components first; the sign itself is evidence worth inspecting.

</details>


## Try It Yourself - move the threshold, order, or prompt

The following cell is deliberately small. Change `PLAY_THRESHOLD` across `0.16`, `0.17`, `0.83`, and `0.84`. Then reverse the deletion order and see whether this graph is order-sensitive. Finally, replace the real clean/corrupt prompt pair with another equal-token-length contrast and rerun the real section.

Before executing, write down a prediction for circuit size and recovery. ARENA-style play should test a hypothesis, not just change a number.


In [ ]:
PLAY_THRESHOLD = 0.17
PLAY_ORDER = deletion_order  # Try tuple(reversed(deletion_order)).

play_result = greedy_acdc(
    edge_names, evaluator,
    clean_metric=clean_metric, corrupt_metric=corrupt_metric,
    threshold=PLAY_THRESHOLD, order=PLAY_ORDER,
)
print({
    "threshold": PLAY_THRESHOLD,
    "kept_edges": len(play_result.kept_edges),
    "recovery": round(play_result.recovery, 3),
    "lost_edges": sorted(set(graph.ground_truth_edges) - set(play_result.kept_edges)),
})


## Bonus - anomaly hunting

Choose one question and turn it into a small result:

1. **Interaction order:** construct a graph with two redundant paths. Does greedy ACDC choose a different minimal circuit when you reverse the candidate order?
2. **Completeness depth:** omit a three-edge path. Show that pairwise completeness still fails, then generalize the subset search.
3. **Negative heads:** `L0H2` has negative deletion damage on the real contrast. Inspect its direct logit attribution and attention pattern. Is it inhibitory, compensatory, or merely prompt-specific?
4. **Prompt transfer:** find a held-out pair where `L0H0/L0H4/L0H6` recovers below `0.75`. Does rediscovery on that pair choose a different set?
5. **MLP mediation:** compare patching the MLP input, MLP output, and selected heads. Which intervention supports an edge claim, and which only identifies a bottleneck?

<details><summary>Help - turn an anomaly into evidence</summary>

State a prediction before running, include a matched control, and report the counterexample even when it weakens the original circuit story.

</details>


## Limitations

- The exact graph is a scalar model organism. It makes ACDC semantics and interaction failures provable, but it is not a transformer.
- The real path is component-level deletion over eight head outputs in one layer. It is not the full ACDC computational graph over Q/K/V, head-to-MLP, and residual edges.
- The real task is a controlled next-token contrast over one primary and three held-out prompt pairs, not a published IOI or greater-than replication.
- The clean target and corrupt distractor are selected from each prompt pair's top predictions. This makes the contrast behaviorally meaningful but narrower than a fixed dataset metric.
- Same-size enumeration is exact because the candidate sets are tiny. Larger models require sampling or structured nulls.
- Pairwise completeness catches the planted backup path but can miss higher-order omitted mechanisms.
- The checked-in verification report records the separate post-rewrite CUDA 13.2 run. The CPU lesson does not invoke that release path.

## Reading Links

- [Towards Automated Circuit Discovery for Mechanistic Interpretability](https://arxiv.org/abs/2304.14997)
- [Attribution Patching Outperforms Automated Circuit Discovery](https://arxiv.org/abs/2310.10348)
- [Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 Small](https://arxiv.org/abs/2211.00593)
- Original ARENA [Indirect Object Identification](../../../chapter1_transformer_interp/instructions/pages/21_[1.4.1]_Indirect_Object_Identification.md)

## Consolidating Understanding

1. An edge score depends on the graph in which the edge is tested.
2. Interacting paths can make every singleton insertion score zero.
3. Greedy ACDC removes one edge, accepts or rejects the deletion, then recomputes the next trial.
4. Threshold sweeps expose the tradeoff between circuit size and behavioral recovery.
5. Faithfulness, minimality, and completeness test different failure modes.
6. Same-size circuits and held-out inputs are controls, not optional decoration.
7. A downstream bottleneck patch is not automatically an upstream edge explanation.


## Notebook Contract

The smoke contract contains the exact toy study, not a placeholder report. It must expose the recovered names, failed one-shot baseline, threshold curve, circuit metrics, all 45 same-size candidates, and held-out regimes.


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict[str, object]:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Full Verification

The lesson above is CPU-runnable. The release path below reruns the exact toy study and the pinned `gelu-1l` component study on CUDA. Leave the flag false while learning; set it true only when you want to regenerate the checked-in evidence on a compatible GPU.


In [ ]:
RUN_FULL_VERIFICATION = False

def run_gpu_test(max_vram_gb: float = 24.0):
    return reference.run_gpu_test(max_vram_gb=max_vram_gb)

def run_full_experiment(max_vram_gb: float = 24.0):
    return run_gpu_test(max_vram_gb=max_vram_gb)

if RUN_FULL_VERIFICATION:
    verification = run_full_experiment()
else:
    verification = json.loads((section_dir / "verification_report.json").read_text())["metrics"]["gpu_test"]

print({
    "device": verification["device"],
    "cuda_version": verification["cuda_version"],
    "real_kept_heads": verification["real_kept_heads"],
    "real_primary_recovery": round(verification["real_primary_recovery"], 3),
    "real_same_size_rank": verification["real_same_size_rank"],
    "peak_vram_gb": round(verification["peak_vram_gb"], 3),
})
